# 🍺 Chopp & Cia · Inteligência de Risco em Comodato
# 🔍 Notebook 02 — Análise Exploratória (EDA)

**Projeto Integrador VI** · 2º Semestre/2026 · FATEC Votorantim

---

## 🎯 Responsabilidade única deste notebook

Entender a carteira e o risco **antes** de modelar. Este notebook **lê e não escreve
dado nenhum** — a única saída são gráficos, tabelas e as conclusões que orientam as
decisões do notebook 03.

| Entrada | Saída |
| :--- | :--- |
| `dataset_consolidado_v<X>` (tabela do notebook 01) | análises, gráficos, conclusões |

## 🧭 O que se decide aqui

A EDA não é ilustração: cada bloco responde a uma pergunta que muda uma escolha
concreta na modelagem.

| Seção | Pergunta | Decisão que informa |
| :--- | :--- | :--- |
| 3.1 Auditoria | o dado fecha? quem está em qual trilha? | confiar ou voltar ao notebook 01 |
| 3.2 Perfil 360° | quem são os clientes e onde está a receita? | quais categóricas viram feature |
| 3.3 Risco por segmento | o risco se concentra em algum grupo? | se `PERFIL`/`CIDADE`/`PAGAMENTO` têm sinal |
| 3.4 Aging | qual a severidade dos atrasos? | onde cortar o limite do alvo |
| 3.5 Recência × risco | quem parou de comprar é mais arriscado? | valor das features de recência |
| 3.6 Matriz de risco | financeiro e comodato andam juntos? | combinador `OU` vs `E` do alvo |
| 3.7 Elegibilidade | a partir de quantas compras a taxa é confiável? | **`min_compras` do notebook 03** |

> A seção 3.7 é a novidade desta versão: ela mede o efeito de `min_compras` sobre
> tamanho da amostra e prevalência do alvo, transformando um número escolhido a dedo
> num número escolhido por evidência.

## Parâmetros de entrada

| Ambiente | Entrada |
| :--- | :--- |
| **Databricks** | tabela versionada publicada pelo notebook 01 |
| **Windows / Jupyter** | CSV consolidado escolhido no seletor de arquivos |

No Databricks, informe `catalogo`, `schema` e `data_version` nos campos do topo.
Localmente, a primeira célula de código abre o seletor do Windows.

In [0]:
# AMBIENTE E PARÂMETROS
from pathlib import Path

try:
    dbutils  # type: ignore[name-defined]
    EM_DATABRICKS = True
except NameError:
    EM_DATABRICKS = False


def widget(nome: str, padrao: str = "", rotulo: str = None, opcoes: list = None) -> str:
    """Widget no Databricks; valor padrão fora dele."""
    if not EM_DATABRICKS:
        return padrao
    try:
        return dbutils.widgets.get(nome)  # type: ignore[name-defined]
    except Exception:
        rotulo = rotulo or nome
        if opcoes:
            dbutils.widgets.dropdown(nome, padrao or opcoes[0], opcoes, rotulo)  # type: ignore[name-defined]
        else:
            dbutils.widgets.text(nome, padrao, rotulo)  # type: ignore[name-defined]
        return dbutils.widgets.get(nome)  # type: ignore[name-defined]


def parametro_arquivo(titulo: str) -> str:
    """Seleciona um CSV somente na execução local."""
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    root.update()
    try:
        caminho = filedialog.askopenfilename(
            parent=root,
            title=titulo,
            filetypes=[("CSV", "*.csv"), ("Todos os arquivos", "*.*")],
        )
    finally:
        root.destroy()
    if not caminho:
        raise ValueError("Nenhum arquivo foi selecionado.")
    return caminho


CSV_ENTRADA = None if EM_DATABRICKS else parametro_arquivo(
    'Selecione o dataset consolidado (CSV do notebook 01)'
)

print(f"Ambiente: {'Databricks' if EM_DATABRICKS else 'Local'}")

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
#  PARÂMETROS DA ANÁLISE
#  Este notebook é somente-leitura: nada aqui altera dado publicado.
# ══════════════════════════════════════════════════════════════════════════════
import os
import warnings
from typing import Dict, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter
from IPython.display import display

# EM_DATABRICKS, widget() e parametro_arquivo() vêm da célula de parametrização
# acima. Ela precisa ter sido executada.
if "parametro_arquivo" not in dir():
    raise NameError(
        "Execute a célula de PARAMETRIZAÇÃO (logo acima) antes desta. "
        "É ela que resolve os caminhos sem deixá-los fixos no código."
    )


# ══════════════════════════════════════════════════════════════════════════════
#  ⬇️  DE ONDE VÊM OS DADOS  ⬇️
#
#  Databricks → tabela versionada publicada pelo notebook 01.
#  Local      → diálogo de seleção do CSV consolidado.
#
#  Nada de caminho fixo: trocar a versão analisada é trocar um campo, e o
#  notebook roda na máquina de qualquer pessoa do grupo sem edição.
# ══════════════════════════════════════════════════════════════════════════════
_CATALOGO = widget("catalogo", "projetointegrador", "Catálogo")
_SCHEMA = widget("schema", "projetointegrador", "Schema")
_DATA_VERSION_ALVO = widget("data_version", "1.0", "Versão dos dados")

TABELA_ENTRADA = (
    f"{_CATALOGO}.{_SCHEMA}.dataset_consolidado_v{_DATA_VERSION_ALVO.replace('.', '_')}"
)

LEITURA_CSV = {"sep": ";", "encoding": "utf-8-sig", "low_memory": False}


# ── Apresentação (sem efeito sobre nenhuma conclusão) ─────────────────────────
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})

CORES_RISCO = {"BAIXO": "#2ECC71", "MÉDIO": "#F39C12", "ALTO": "#E74C3C"}
COR_PRIMARIA, COR_SECUNDARIA, COR_ALERTA = "#3498DB", "#2ECC71", "#E74C3C"

# Ordem canônica das faixas de aging. Aging é categórica ORDINAL: sem declarar a
# ordem, os gráficos saem em ordem alfabética e "1-3 Dias" aparece depois de
# "+30 Dias", o que inverte a leitura de severidade.
ORDEM_AGING = ["Sem Atraso", "1-3 Dias", "4-7 Dias", "8-15 Dias",
               "16-20 Dias", "21-30 Dias", "+30 Dias"]


def fmt_moeda(x, pos=None):
    """Formata valores como moeda brasileira abreviada."""
    if x >= 1e6:
        return f"R$ {x*1e-6:.1f}M"
    if x >= 1e3:
        return f"R$ {x*1e-3:.1f}k"
    return f"R$ {x:.0f}"


def fmt_numero(x, pos=None):
    """Formata inteiros com separador de milhar."""
    return f"{int(x):,}".replace(",", ".")


print("=" * 78)
print(f"{'PARÂMETROS DA EDA':^78}")
print("=" * 78)
print(f"  Ambiente : {'Databricks' if EM_DATABRICKS else 'Local (CSV)'}")
print(f"  Entrada  : {TABELA_ENTRADA if EM_DATABRICKS else os.path.basename(CSV_ENTRADA)}")
print("=" * 78)

## 📂 2. Carga e Verificação do Contrato

A tabela Delta já traz o schema — `TOTAL_GASTO` volta como `double`, datas como
`timestamp`. A normalização de tipos permanece mesmo assim: esta célula precisa
produzir **o mesmo `df`** tendo vindo da tabela ou do CSV, senão as duas origens
divergem em silêncio e o resultado depende de onde o notebook rodou.

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
#  CARGA DO DATASET CONSOLIDADO
# ══════════════════════════════════════════════════════════════════════════════
print(f"📂 Carregando {TABELA_ENTRADA if EM_DATABRICKS else os.path.basename(CSV_ENTRADA)}...\n")

if EM_DATABRICKS:
    df = spark.read.table(TABELA_ENTRADA).toPandas()  # type: ignore[name-defined]
else:
    df = pd.read_csv(CSV_ENTRADA, **LEITURA_CSV)

# ── Normalização de tipos ─────────────────────────────────────────────────────
for _col in ["PRIMEIRA_COMPRA", "ULTIMA_COMPRA"]:
    if _col in df.columns:
        df[_col] = pd.to_datetime(df[_col], errors="coerce")

_COLS_NUM = [
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA", "FREQUENCIA_COMPRAS",
    "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS",
    "RISCO_FINANCEIRO", "RISCO_COMODATO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
]
for _col in _COLS_NUM:
    if _col in df.columns:
        df[_col] = pd.to_numeric(df[_col], errors="coerce").fillna(0)

for _col in ["AGING_PAGAMENTO", "AGING_COMODATO"]:
    if _col in df.columns:
        df[_col] = pd.Categorical(df[_col], categories=ORDEM_AGING, ordered=True)

# ── Contrato ──────────────────────────────────────────────────────────────────
# Falhar aqui é muito mais barato que uma coluna faltando trinta células adiante.
_ESPERADAS = [
    "ID_PESSOA", "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO",
    "FREQUENCIA_COMPRAS", "TICKET_MEDIO", "TOTAL_GASTO",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "TOTAL_PARCELAS", "TOTAL_COMODATOS",
    "TAXA_ATRASO_PAGAMENTO", "TAXA_ATRASO_COMODATO",
    "AGING_PAGAMENTO", "AGING_COMODATO", "PERFIL_RISCO", "CORE_BUSINESS",
]
_ausentes = [c for c in _ESPERADAS if c not in df.columns]
if _ausentes:
    raise KeyError(
        f"Colunas ausentes em {TABELA_ENTRADA}: {_ausentes}.\n"
        f"   O contrato mudou — reexecute o notebook 01 com DATA_VERSION nova."
    )

if df["ID_PESSOA"].duplicated().any():
    raise ValueError("ID_PESSOA duplicado — a tabela deveria ter 1 linha por cliente.")

# ── Proveniência: qual versão estamos analisando ──────────────────────────────
DATA_VERSION_LIDA = (
    str(df["_DATA_VERSION"].iloc[0]) if "_DATA_VERSION" in df.columns else "desconhecida"
)
INGESTAO_HASH_LIDO = (
    str(df["_INGESTAO_HASH"].iloc[0]) if "_INGESTAO_HASH" in df.columns else "desconhecido"
)

print(f"   ✅ {df.shape[0]:,} clientes × {df.shape[1]} colunas")
print(f"   ├─ Versão dos dados : {DATA_VERSION_LIDA}")
print(f"   ├─ Hash da ingestão : {INGESTAO_HASH_LIDO}")
print(f"   └─ Janela           : {df['PRIMEIRA_COMPRA'].min():%d/%m/%Y} → "
      f"{df['ULTIMA_COMPRA'].max():%d/%m/%Y}")

display(df.head())

## 📊 3.1 — Auditoria da Base e Cobertura das Trilhas

Duas perguntas antes de qualquer gráfico: **o dado fecha?** e **quem está em qual
trilha?**

As flags `TEM_VENDAS` / `TEM_FINANCEIRO` / `TEM_COMODATO` distinguem "não tem
comodato" de "comodato não encontrado" — leituras muito diferentes que um zero
sozinho confunde. A matriz de combinações mostra as 2³ possibilidades e revela
qualquer célula inesperada, como comodato sem nenhuma venda associada.

> **Taxas ponderadas pelo volume:** somamos numerador e denominador de toda a
> carteira em vez de tirar a média das taxas individuais. Um cliente com 1 parcela
> não deve pesar o mesmo que um com 200.

In [0]:
# ─── 3.1 Auditoria da base consolidada ────────────────────────────────────────
print("=" * 68)
print(f"{'📊 3.1 · AUDITORIA DA BASE CONSOLIDADA':^68}")
print("=" * 68)

n_clientes = len(df)
print(f"\n👥 Clientes únicos (ID_PESSOA)      : {n_clientes:>6,}")
for _flag, _rot in [("TEM_VENDAS", "com histórico de vendas"),
                    ("TEM_FINANCEIRO", "com trilha financeira"),
                    ("TEM_COMODATO", "com comodato"),
                    ("CORE_BUSINESS", "no core business (chopp)")]:
    _n = int(df[_flag].sum())
    print(f"   ├─ {_rot:<30}: {_n:>6,}  ({_n/n_clientes*100:>5.1f}%)")

# ── Pré-diagnóstico de atraso, ponderado pelo volume ──────────────────────────
tot_parcelas = df["TOTAL_PARCELAS"].sum()
tot_parc_atraso = df["PARCELAS_ATRASADAS"].sum()
tot_comodatos = df["TOTAL_COMODATOS"].sum()
tot_com_atraso = df["COMODATOS_ATRASADOS"].sum()

pct_fin = (tot_parc_atraso / tot_parcelas * 100) if tot_parcelas else 0
pct_com = (tot_com_atraso / tot_comodatos * 100) if tot_comodatos else 0

print(f"\n{'─'*68}")
print(f"{'⚠️  PRÉ-DIAGNÓSTICO DE ATRASOS (ponderado por volume)':^68}")
print(f"{'─'*68}")
print(f"   🔴 Atraso financeiro : {pct_fin:>5.1f}%  "
      f"({tot_parc_atraso:,.0f} de {tot_parcelas:,.0f} parcelas)")
print(f"   🟠 Atraso comodato   : {pct_com:>5.1f}%  "
      f"({tot_com_atraso:,.0f} de {tot_comodatos:,.0f} contratos)")

# ── Matriz de combinações das trilhas ─────────────────────────────────────────
flags = ["TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO"]
combos = (
    df.groupby(flags).size().reset_index(name="QTD")
    .sort_values("QTD", ascending=False)
)

print(f"\n{'─'*68}")
print(f"{'👀 MATRIZ DE COBERTURA (Vendas / Financeiro / Comodato)':^68}")
print(f"{'─'*68}")
_marca = lambda v: "✔" if v == 1 else "·"
for _, r in combos.iterrows():
    print(f"   V {_marca(r['TEM_VENDAS'])} | F {_marca(r['TEM_FINANCEIRO'])} "
          f"| C {_marca(r['TEM_COMODATO'])}  →  {int(r['QTD']):>5,} clientes")

_completa = int(((df[flags] == 1).all(axis=1)).sum())
print(f"\n🎯 Trilha completa (V+F+C): {_completa:,} clientes "
      f"({_completa/n_clientes*100:.1f}% da carteira)")

In [0]:
# ─── 3.1b Top 10 clientes por faturamento ─────────────────────────────────────
# O ranking nasce pronto: o dataset já chega com 1 linha por ID_PESSOA e as
# métricas de RFM consolidadas no notebook 01.
#
# Identificação por ID_PESSOA : as colunas de nome saíram do pipeline.
# Para a leitura que interessa aqui — o quanto o faturamento se concentra e que
# perfil tem quem está no topo — o identificador basta. Se for preciso saber
# QUEM é o cliente 1042, a consulta é no ERP.
top10 = df.sort_values("TOTAL_GASTO", ascending=False).head(10)[
    ["ID_PESSOA", "PERFIL", "CIDADE", "SEGMENTO", "PRODUTO_FAVORITO",
     "FREQUENCIA_COMPRAS", "TOTAL_GASTO", "TICKET_MEDIO"]
].copy()

# Blindagem contra cadastro incompleto — a origem pode trazer campos nulos.
top10["SEGMENTO"] = top10["SEGMENTO"].fillna("OUTROS")
top10["PRODUTO_FAVORITO"] = top10["PRODUTO_FAVORITO"].fillna("N/D")

top10.columns = ["ID do Cliente", "Perfil", "Cidade", "Segmento",
                 "Produto Mais Comprado", "Total de Pedidos",
                 "Total Comprado (R$)", "Ticket Médio (R$)"]

display(
    top10.style
    .format({"Total Comprado (R$)": "R$ {:,.2f}", "Ticket Médio (R$)": "R$ {:,.2f}"})
    .hide(axis="index")
    .set_caption("3.1 · Top 10 Clientes por Volume de Compras")
)

# ── Concentração de receita ───────────────────────────────────────────────────
# A pergunta de negócio que o ranking existe para responder: a carteira depende
# de poucos clientes? Se depende, o custo de errar num deles é desproporcional —
# e isso muda como se lê um falso negativo do modelo.
_fat = df["TOTAL_GASTO"].sort_values(ascending=False)
_total = _fat.sum()
print("=" * 68)
print(f"{'CONCENTRAÇÃO DE RECEITA':^68}")
print("=" * 68)
for _n in [5, 10, 20, 50]:
    if _n <= len(_fat):
        _pct = _fat.head(_n).sum() / _total * 100
        print(f"   Top {_n:>3} clientes ({_n/len(_fat)*100:>4.1f}% da carteira) "
              f"→ {_pct:>5.1f}% do faturamento")

### 💡 Conclusões da auditoria

1. **Inconsistência na segmentação cadastral.** O `SEGMENTO` do cadastro não reflete
   de forma confiável a realidade do cliente — a auditoria manual feita na v1 do
   projeto encontrou estabelecimentos comerciais de grande porte classificados como
   "Consumidor Final". A célula abaixo quantifica o sintoma sem depender do nome:
   clientes rotulados como consumidor final que compram em volume e frequência de
   estabelecimento comercial. Por isso `SEGMENTO` **não entra como feature** no
   notebook 03 — treinar sobre um rótulo sabidamente errado ensina o erro ao modelo.

2. **Pedidos de venda e de comodato têm `ID_PEDIDO` distintos no ERP**, por razão
   fiscal: um pedido gera receita (venda de chopp), outro é remessa de ativo
   (comodato da chopeira). Consequência metodológica: o cruzamento entre as trilhas
   é feito no nível do **cliente** (`ID_PESSOA`), nunca no da transação — e é por
   isso que todo o pipeline usa `ID_PESSOA` como unidade de análise.

In [0]:
# ─── 3.1c Evidência do ruído em SEGMENTO ──────────────────────────────────────
# Sem os nomes , o sintoma é medido pelo COMPORTAMENTO em vez de lido no
# cadastro: um "Consumidor Final" que compra como estabelecimento comercial é a
# marca do rótulo errado. O critério é o próprio dado — mediana de frequência e
# faturamento dos segmentos declaradamente comerciais.
_SEG_CF = ["CONSUMIDOR FINAL", "CONSUMIDOR", "PESSOA FISICA", "PESSOA FÍSICA"]
_mask_cf = df["SEGMENTO"].str.upper().isin([s.upper() for s in _SEG_CF])

print("=" * 74)
print(f"{'RUÍDO NO CADASTRO DE SEGMENTO':^74}")
print("=" * 74)

if _mask_cf.sum() == 0:
    print("\n  Nenhum cliente rotulado como consumidor final nesta carga —")
    print("  a checagem não se aplica. SEGMENTO segue fora das features por")
    print("  decisão de projeto (ver conclusão 1).")
else:
    # Referência: o comportamento típico de quem é declaradamente comercial
    _com = df[~_mask_cf]
    _lim_freq = _com["FREQUENCIA_COMPRAS"].median()
    _lim_fat = _com["TOTAL_GASTO"].median()

    _cf = df[_mask_cf]
    _suspeitos = _cf[(_cf["FREQUENCIA_COMPRAS"] > _lim_freq)
                     & (_cf["TOTAL_GASTO"] > _lim_fat)]

    print(f"\n  Clientes rotulados 'consumidor final'      : {len(_cf):>5,}")
    print(f"  Mediana dos comerciais (freq / faturamento): "
          f"{_lim_freq:.0f} pedidos / R$ {_lim_fat:,.0f}")
    print(f"  Desses, compram ACIMA das duas medianas    : {len(_suspeitos):>5,} "
          f"({len(_suspeitos)/max(len(_cf),1)*100:.1f}%)")

    if len(_suspeitos):
        print(f"\n  Faturamento concentrado nos suspeitos      : "
              f"R$ {_suspeitos['TOTAL_GASTO'].sum():,.2f}")
        print(f"\n  ➜ Um consumidor final que compra mais que a mediana dos")
        print(f"     estabelecimentos comerciais quase certamente é um comércio")
        print(f"     mal cadastrado. É por isso que SEGMENTO não vira feature:")
        print(f"     o modelo aprenderia o erro do cadastro, não o risco.")
    else:
        print(f"\n  ➜ Nenhum caso flagrante nesta carga. SEGMENTO segue fora das")
        print(f"     features por decisão de projeto — o rótulo é preenchido por")
        print(f"     digitação manual e não tem garantia de consistência.")
print("=" * 74)

## 📊 3.2 — Perfil do Cliente (Visão 360°)

Três dimensões cadastrais de alta confiabilidade, cruzadas com três métricas de
negócio (pedidos, receita, ticket médio):

1. **Perfil** — Física / Jurídica / Estrangeiro. Isola o comportamento B2B, o
   consumidor final e o público transfronteiriço (fronteira seca com o Paraguai).
2. **Cidade** — onde está a demanda, e portanto o risco logístico de recolher
   equipamento.
3. **Forma de pagamento** — o método de acerto padrão, candidato natural a preditor
   de inadimplência.

> `SEGMENTO` fica fora deste painel pelo motivo da conclusão anterior: o cadastro é
> ruidoso demais para sustentar leitura.

In [0]:
# ─── 3.2 Perfil 360° ──────────────────────────────────────────────────────────
df_360 = df.copy()
df_360["CIDADE"] = df_360["CIDADE"].fillna("NÃO PREENCHIDO")

DIMENSOES = [
    ("PERFIL", "Perfil do Cliente"),
    ("CIDADE", "Localização Geográfica"),
    ("PAGAMENTO", "Forma de Pagamento"),
]
TOP_N = 6   # limita categorias de alta cardinalidade à cauda relevante


def _rotular_barras(ax, formatador=None):
    """Anota o valor no topo de cada barra."""
    for p in ax.patches:
        val = p.get_height()
        if val and val > 0:
            texto = formatador(val) if formatador else f"{int(val):,}"
            ax.annotate(texto, (p.get_x() + p.get_width() / 2, val),
                        ha="center", va="bottom", fontsize=8, fontweight="bold")


fig, axes = plt.subplots(len(DIMENSOES), 3, figsize=(18, 5 * len(DIMENSOES)))
fig.suptitle("3.2 · Perfil 360° da Carteira", fontsize=17, fontweight="bold", y=0.995)

for linha, (coluna, titulo) in enumerate(DIMENSOES):
    # Agregação no grão do cliente: pedidos somam FREQUENCIA_COMPRAS, receita soma
    # TOTAL_GASTO, e o ticket médio é a razão entre os dois — não a média dos
    # tickets individuais, que daria peso igual a clientes de tamanhos diferentes.
    agg = (
        df_360.groupby(coluna)
        .agg(PEDIDOS=("FREQUENCIA_COMPRAS", "sum"),
             FATURAMENTO=("TOTAL_GASTO", "sum"),
             CLIENTES=("ID_PESSOA", "count"))
        .sort_values("FATURAMENTO", ascending=False)
        .head(TOP_N)
    )
    agg["TICKET"] = agg["FATURAMENTO"] / agg["PEDIDOS"].replace(0, np.nan)

    for col, (metrica, rotulo, cor, fmt) in enumerate([
        ("PEDIDOS", "Pedidos", COR_PRIMARIA, None),
        ("FATURAMENTO", "Faturamento", COR_SECUNDARIA, fmt_moeda),
        ("TICKET", "Ticket Médio", "#9B59B6", fmt_moeda),
    ]):
        ax = axes[linha, col]
        agg[metrica].plot(kind="bar", ax=ax, color=cor, edgecolor="white", alpha=0.9)
        ax.set_title(f"{titulo} · {rotulo}", fontsize=11, fontweight="bold")
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=35, labelsize=8)
        for lbl in ax.get_xticklabels():
            lbl.set_ha("right")
        if fmt:
            ax.yaxis.set_major_formatter(FuncFormatter(fmt))
        _rotular_barras(ax, fmt)

plt.tight_layout()
plt.show()

# Resumo textual: complementa o gráfico com o número de clientes por categoria,
# que a escala de faturamento esconde.
for coluna, titulo in DIMENSOES:
    _vc = df_360[coluna].value_counts().head(TOP_N)
    print(f"\n{titulo} ({df_360[coluna].nunique()} categorias):")
    for cat, n in _vc.items():
        print(f"   {str(cat)[:38]:<40} {n:>5,} clientes ({n/len(df_360)*100:>5.1f}%)")

## ⚠️ 3.3 — Risco por Dimensão Cadastral

**A pergunta que decide se `PERFIL`, `CIDADE` e `PAGAMENTO` merecem ser features:**
a taxa de risco varia entre as categorias, ou é a mesma em todo lugar?

Uma dimensão cujo risco é plano não informa nada ao modelo — ela só acrescenta
colunas ao one-hot e dilui os dados. Uma dimensão com variação forte é onde o modelo
vai buscar sinal.

> A unidade aqui é o **cliente**, não o pedido: a média de `RISCO_FINANCEIRO` num
> grupo é a fração da carteira daquele grupo sinalizada como arriscada.

In [0]:
# ─── 3.3 Risco por dimensão cadastral ─────────────────────────────────────────
fig, axes = plt.subplots(len(DIMENSOES), 1, figsize=(13, 5 * len(DIMENSOES)))
fig.suptitle("3.3 · Taxa de Risco por Dimensão Cadastral",
             fontsize=16, fontweight="bold", y=0.997)

# Linha de referência: a taxa média da carteira. Sem ela não se sabe se 25% é
# muito ou pouco — é a âncora que transforma o gráfico em leitura comparativa.
media_fin = df["RISCO_FINANCEIRO"].mean() * 100
media_com = df["RISCO_COMODATO"].mean() * 100

resumo_dimensoes = {}

for i, (coluna, titulo) in enumerate(DIMENSOES):
    agg = (
        df.groupby(coluna)
        .agg(CLIENTES=("ID_PESSOA", "count"),
             RISCO_FIN=("RISCO_FINANCEIRO", "mean"),
             RISCO_COM=("RISCO_COMODATO", "mean"))
        .sort_values("CLIENTES", ascending=False)
        .head(TOP_N)
    )
    agg[["RISCO_FIN", "RISCO_COM"]] *= 100
    resumo_dimensoes[coluna] = agg

    ax = axes[i]
    x = np.arange(len(agg))
    largura = 0.38
    ax.bar(x - largura/2, agg["RISCO_FIN"], largura,
           label="Risco Financeiro", color=COR_ALERTA, alpha=0.88)
    ax.bar(x + largura/2, agg["RISCO_COM"], largura,
           label="Risco Comodato", color="#F39C12", alpha=0.88)

    ax.axhline(media_fin, ls="--", lw=1.2, color=COR_ALERTA, alpha=0.55,
               label=f"Média financeiro ({media_fin:.0f}%)")
    ax.axhline(media_com, ls=":", lw=1.2, color="#F39C12", alpha=0.55,
               label=f"Média comodato ({media_com:.0f}%)")

    ax.set_xticks(x)
    ax.set_xticklabels([str(v)[:24] for v in agg.index], rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("% de clientes em risco")
    ax.set_title(f"{titulo}  ·  n por categoria à direita das barras",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc="upper right")
    ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))

    # O n de cada categoria é indispensável: 100% de risco em 3 clientes não é
    # o mesmo achado que 40% em 300.
    for j, (_, r) in enumerate(agg.iterrows()):
        ax.annotate(f"n={int(r['CLIENTES'])}",
                    (j, max(r["RISCO_FIN"], r["RISCO_COM"])),
                    ha="center", va="bottom", fontsize=7.5, color="#555")

plt.tight_layout()
plt.show()

# ── Leitura quantitativa: amplitude do risco dentro de cada dimensão ──────────
# É a amplitude — e não o nível — que diz se a dimensão discrimina.
print("=" * 74)
print(f"{'PODER DISCRIMINANTE DAS DIMENSÕES (amplitude da taxa de risco)':^74}")
print("=" * 74)
print(f"\n{'Dimensão':<14} {'Amplitude fin.':>15} {'Amplitude com.':>15}   Leitura")
print("-" * 74)
for coluna, agg in resumo_dimensoes.items():
    amp_f = agg["RISCO_FIN"].max() - agg["RISCO_FIN"].min()
    amp_c = agg["RISCO_COM"].max() - agg["RISCO_COM"].min()
    leitura = "discrimina" if max(amp_f, amp_c) > 15 else "pouco sinal"
    print(f"{coluna:<14} {amp_f:>14.1f}p {amp_c:>14.1f}p   {leitura}")
print("-" * 74)
print("  Amplitude = maior taxa − menor taxa entre as categorias da dimensão.")
print("  Amplitude baixa ⇒ a dimensão só acrescenta colunas ao one-hot sem trazer sinal.")

## 📅 3.4 — Aging da Carteira

A distribuição da **severidade** dos atrasos, contada em clientes. As faixas usam o
atraso **máximo** de cada cliente, não o médio: quem teve um atraso de 45 dias é caso
de "+30 Dias" ainda que a média o diluísse numa faixa branda.

Esta é a leitura que informa **onde cortar o limite do alvo** no notebook 03: se a
massa está em "Sem Atraso" e "+30 Dias", com pouco no meio, o problema é mais binário
do que gradual — e um limite de 20% separa bem.

In [0]:
# ─── 3.4 Aging da carteira ────────────────────────────────────────────────────
# As colunas são pd.Categorical ordenadas: value_counts(sort=False) já devolve na
# ordem cronológica das faixas, sem reindexar na mão.
aging_fin = df["AGING_PAGAMENTO"].value_counts(sort=False)
aging_com = df["AGING_COMODATO"].value_counts(sort=False)
df_aging = pd.DataFrame({"Financeiro": aging_fin, "Comodato": aging_com}).fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
fig.suptitle("3.4 · Aging da Carteira (clientes por faixa do PIOR atraso)",
             fontsize=15, fontweight="bold")

# (a) Contagem absoluta lado a lado
ax = axes[0]
df_aging.plot(kind="bar", ax=ax, color=[COR_ALERTA, "#F39C12"],
              edgecolor="white", alpha=0.9)
ax.set_title("Clientes por faixa", fontsize=12, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("clientes")
ax.tick_params(axis="x", rotation=35, labelsize=8.5)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
ax.legend(fontsize=9)

# (b) Distribuição percentual: compara as duas trilhas apesar dos denominadores
#     diferentes (nem todo cliente tem comodato)
ax = axes[1]
(df_aging / df_aging.sum() * 100).plot(kind="bar", ax=ax,
                                       color=[COR_ALERTA, "#F39C12"],
                                       edgecolor="white", alpha=0.9)
ax.set_title("Distribuição percentual", fontsize=12, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("% dos clientes da trilha")
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.tick_params(axis="x", rotation=35, labelsize=8.5)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
ax.legend(fontsize=9)

# (c) Acumulado invertido: "quantos clientes têm atraso de ao menos X dias"
#     É a curva que um gestor de crédito lê para calibrar política de cobrança.
ax = axes[2]
for nome, cor in [("Financeiro", COR_ALERTA), ("Comodato", "#F39C12")]:
    serie = df_aging[nome]
    acum = serie[::-1].cumsum()[::-1] / serie.sum() * 100
    ax.plot(range(len(acum)), acum.values, marker="o", lw=2.2, color=cor, label=nome)
ax.set_xticks(range(len(ORDEM_AGING)))
ax.set_xticklabels(ORDEM_AGING, rotation=35, ha="right", fontsize=8.5)
ax.set_title("Acumulado: % com atraso ≥ faixa", fontsize=12, fontweight="bold")
ax.set_ylabel("% dos clientes")
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 68)
print(f"{'AGING · LEITURA NUMÉRICA':^68}")
print("=" * 68)
print(f"\n{'Faixa':<14} {'Financeiro':>12} {'%':>8} {'Comodato':>12} {'%':>8}")
print("-" * 68)
for faixa in ORDEM_AGING:
    f = int(df_aging.loc[faixa, "Financeiro"]) if faixa in df_aging.index else 0
    c = int(df_aging.loc[faixa, "Comodato"]) if faixa in df_aging.index else 0
    print(f"{faixa:<14} {f:>12,} {f/max(df_aging['Financeiro'].sum(),1)*100:>7.1f}% "
          f"{c:>12,} {c/max(df_aging['Comodato'].sum(),1)*100:>7.1f}%")

_sem_atraso_fin = df_aging.loc["Sem Atraso", "Financeiro"] / df_aging["Financeiro"].sum() * 100
_grave_fin = df_aging.loc["+30 Dias", "Financeiro"] / df_aging["Financeiro"].sum() * 100
print("-" * 68)
print(f"  Sem atraso algum : {_sem_atraso_fin:.1f}% da carteira")
print(f"  Atraso grave     : {_grave_fin:.1f}% acima de 30 dias")

## 📈 3.5 — Recência × Risco

**A hipótese central do modelo:** cliente que parou de comprar é mais arriscado?

Se a resposta for sim, `DIAS_DESDE_ULTIMA_COMPRA` carrega sinal genuíno — e é uma das
poucas features do conjunto que **não** compartilha origem aritmética com o alvo.

> ⚠️ **Limitação do grão.** A série de faturamento mês a mês exigiria `DT_PEDIDO` por
> transação. O dataset guarda apenas `MES_ULTIMA_COMPRA`, então cada cliente aparece
> **uma vez**, no seu mês de recência. A leitura é "curva de recência da carteira",
> não "curva de receita" — não confunda uma com a outra ao apresentar.

In [0]:
# ─── 3.5 Recência × risco ─────────────────────────────────────────────────────
df_rec = df[df["TEM_VENDAS"] == 1].copy()
df_rec["ANO_MES"] = df_rec["MES_ULTIMA_COMPRA"].astype(str)
df_rec = df_rec[df_rec["ANO_MES"].str.match(r"^\d{4}-\d{2}$", na=False)]

serie = (
    df_rec.groupby("ANO_MES")
    .agg(CLIENTES=("ID_PESSOA", "count"),
         FATURAMENTO=("TOTAL_GASTO", "sum"),
         RISCO_FIN=("RISCO_FINANCEIRO", "mean"),
         RISCO_COM=("RISCO_COMODATO", "mean"))
    .sort_index()
)
serie[["RISCO_FIN", "RISCO_COM"]] *= 100

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
fig.suptitle("3.5 · Recência da Carteira × Risco", fontsize=15, fontweight="bold")

# (a) Quantos clientes têm sua última compra em cada mês
ax = axes[0]
ax.bar(range(len(serie)), serie["CLIENTES"], color=COR_PRIMARIA, alpha=0.85)
ax.set_ylabel("clientes cuja ÚLTIMA compra foi no mês")
ax.set_title("Distribuição da recência", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# (b) O risco desses clientes — a pergunta que interessa
ax = axes[1]
ax.plot(range(len(serie)), serie["RISCO_FIN"], marker="o", lw=2.2,
        color=COR_ALERTA, label="Risco financeiro")
ax.plot(range(len(serie)), serie["RISCO_COM"], marker="s", lw=2.2,
        color="#F39C12", label="Risco comodato")
ax.axhline(media_fin, ls="--", lw=1, color=COR_ALERTA, alpha=0.5)
ax.axhline(media_com, ls=":", lw=1, color="#F39C12", alpha=0.5)
ax.set_ylabel("% em risco")
ax.set_title("Risco por mês da última compra (abandono mais antigo → à esquerda)",
             fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

_passo = max(1, len(serie) // 24)
ax.set_xticks(range(0, len(serie), _passo))
ax.set_xticklabels(serie.index[::_passo], rotation=60, ha="right", fontsize=7.5)

plt.tight_layout()
plt.show()

# ── Teste direto da hipótese: risco por quartil de recência ──────────────────
# O gráfico mensal é ruidoso nas caudas (poucos clientes por mês). Agrupar em
# quartis dá a leitura estável da mesma pergunta.
_q = df_rec.copy()
_q["QUARTIL_RECENCIA"] = pd.qcut(
    _q["DIAS_DESDE_ULTIMA_COMPRA"], 4,
    labels=["Q1 · mais recentes", "Q2", "Q3", "Q4 · mais antigos"],
)
tab = (
    _q.groupby("QUARTIL_RECENCIA", observed=True)
    .agg(CLIENTES=("ID_PESSOA", "count"),
         DIAS_MEDIANOS=("DIAS_DESDE_ULTIMA_COMPRA", "median"),
         RISCO_FIN=("RISCO_FINANCEIRO", "mean"),
         RISCO_COM=("RISCO_COMODATO", "mean"))
)
tab[["RISCO_FIN", "RISCO_COM"]] = (tab[["RISCO_FIN", "RISCO_COM"]] * 100).round(1)

print("=" * 78)
print(f"{'HIPÓTESE: QUEM PAROU DE COMPRAR É MAIS ARRISCADO?':^78}")
print("=" * 78)
print(f"\n{'Quartil de recência':<22} {'Clientes':>9} {'Dias (med.)':>12} "
      f"{'Risco fin.':>11} {'Risco com.':>11}")
print("-" * 78)
for idx, r in tab.iterrows():
    print(f"{str(idx):<22} {int(r['CLIENTES']):>9,} {r['DIAS_MEDIANOS']:>12,.0f} "
          f"{r['RISCO_FIN']:>10.1f}% {r['RISCO_COM']:>10.1f}%")

_delta_fin = tab["RISCO_FIN"].iloc[-1] - tab["RISCO_FIN"].iloc[0]
_delta_com = tab["RISCO_COM"].iloc[-1] - tab["RISCO_COM"].iloc[0]
print("-" * 78)
print(f"  Diferença Q4 − Q1 : financeiro {_delta_fin:+.1f}p · comodato {_delta_com:+.1f}p")
if max(abs(_delta_fin), abs(_delta_com)) > 10:
    print("  ✅ A recência discrimina risco — DIAS_DESDE_ULTIMA_COMPRA tem sinal próprio.")
else:
    print("  ⚠️  Recência com pouca variação de risco: a feature pode não contribuir.")

## 🎯 3.6 — Matriz de Risco Integrado

**A pergunta que decide o combinador do alvo:** inadimplência financeira e retenção
de equipamento andam juntas?

- Se andam **juntas**, `E` e `OU` selecionam quase o mesmo grupo — e a escolha
  importa pouco.
- Se são **independentes**, `OU` produz um alvo bem mais prevalente que `E`, e a
  decisão muda materialmente quantos clientes são rotulados como alto risco.

O notebook 03 declara esse combinador no painel; esta seção é a evidência que sustenta
a escolha.

In [0]:
# ─── 3.6 Matriz de risco integrado ────────────────────────────────────────────
matriz = pd.crosstab(
    df["RISCO_FINANCEIRO"], df["RISCO_COMODATO"],
    rownames=["Risco Financeiro"], colnames=["Risco Comodato"],
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle("3.6 · Risco Financeiro × Risco de Comodato",
             fontsize=15, fontweight="bold")

ax = axes[0]
sns.heatmap(matriz, annot=True, fmt=",d", cmap="Reds", ax=ax,
            cbar_kws={"label": "clientes"}, linewidths=1.4, linecolor="white")
ax.set_title("Matriz de contingência (clientes)", fontsize=12, fontweight="bold")
ax.set_xticklabels(["Sem risco", "Com risco"])
ax.set_yticklabels(["Sem risco", "Com risco"], rotation=0)

ax = axes[1]
perfil = df["PERFIL_RISCO"].value_counts()
_cores = {"SEM RISCO": COR_SECUNDARIA, "SÓ FINANCEIRO": COR_ALERTA,
          "SÓ COMODATO": "#F39C12", "RISCO DUPLO": "#8E44AD"}
ax.bar(range(len(perfil)), perfil.values,
       color=[_cores.get(k, "#7F8C8D") for k in perfil.index],
       edgecolor="white", alpha=0.9)
ax.set_xticks(range(len(perfil)))
ax.set_xticklabels(perfil.index, rotation=18, ha="right", fontsize=9.5)
ax.set_ylabel("clientes")
ax.set_title("Perfil de risco integrado", fontsize=12, fontweight="bold")
for i, v in enumerate(perfil.values):
    ax.annotate(f"{v:,}\n({v/len(df)*100:.1f}%)", (i, v),
                ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

# ── Independência entre as trilhas ────────────────────────────────────────────
# Compara a co-ocorrência observada com a esperada sob independência. Um lift
# próximo de 1 diz que uma trilha não prevê a outra — e nesse caso 'OU' e 'E'
# selecionam grupos genuinamente diferentes.
p_fin = df["RISCO_FINANCEIRO"].mean()
p_com = df["RISCO_COMODATO"].mean()
p_ambos_obs = ((df["RISCO_FINANCEIRO"] == 1) & (df["RISCO_COMODATO"] == 1)).mean()
p_ambos_esp = p_fin * p_com
lift = p_ambos_obs / p_ambos_esp if p_ambos_esp > 0 else float("nan")

n_ou = int(((df["RISCO_FINANCEIRO"] == 1) | (df["RISCO_COMODATO"] == 1)).sum())
n_e = int(((df["RISCO_FINANCEIRO"] == 1) & (df["RISCO_COMODATO"] == 1)).sum())

print("=" * 78)
print(f"{'AS DUAS TRILHAS DE RISCO ANDAM JUNTAS?':^78}")
print("=" * 78)
print(f"\n  P(risco financeiro)          : {p_fin:>7.1%}")
print(f"  P(risco comodato)            : {p_com:>7.1%}")
print(f"  P(ambos) observado           : {p_ambos_obs:>7.1%}")
print(f"  P(ambos) se independentes    : {p_ambos_esp:>7.1%}")
print(f"  Lift (observado / esperado)  : {lift:>7.2f}×")
print("-" * 78)
if lift > 1.3:
    print("  ➜ As trilhas se REFORÇAM: quem atrasa pagamento tende a reter equipamento.")
elif lift < 0.8:
    print("  ➜ As trilhas se EXCLUEM: são populações de risco distintas.")
else:
    print("  ➜ As trilhas são praticamente INDEPENDENTES.")

print(f"\n  Efeito do combinador sobre o alvo do notebook 03:")
print(f"     combinador 'OU' → {n_ou:>5,} clientes em risco ({n_ou/len(df):>5.1%})")
print(f"     combinador 'E'  → {n_e:>5,} clientes em risco ({n_e/len(df):>5.1%})")
print(f"\n  ⚠️  São problemas DIFERENTES: métricas de rodadas com combinadores")
print(f"      distintos não são comparáveis entre si.")

## 🔬 3.7 — Elegibilidade: quantas compras um cliente precisa ter?

**Esta seção existe para responder a uma pergunta que antes era decidida a dedo:**
qual valor de `min_compras` usar no notebook 03?

O problema é real. A taxa de atraso de um cliente com **uma** parcela só pode valer
0% ou 100% — não existe meio-termo. Ela entra no dataset com a mesma aparência de uma
taxa calculada sobre 200 parcelas, e o modelo trata as duas como igualmente
informativas. Filtrar por histórico mínimo remove esse ruído; filtrar demais joga
fora metade da amostra.

O trade-off é explícito:

| `min_compras` ↑ | `min_compras` ↓ |
| :--- | :--- |
| taxas mais confiáveis | mais clientes na amostra |
| menos clientes | intervalos de confiança mais estreitos |
| menos ruído de denominador pequeno | mais ruído de denominador pequeno |

A tabela e o gráfico abaixo mostram o custo de cada corte. **O notebook 04 varre esse
mesmo eixo como experimento no MLflow** — aqui você vê a estrutura dos dados; lá,
o efeito sobre a métrica do modelo.

In [0]:
# ══════════════════════════════════════════════════════════════════════════════
#  3.7 · ESTUDO DE ELEGIBILIDADE — o efeito de min_compras
#
#  Alimenta a escolha de HP_ARQUITETURA['dados']['min_compras'] no notebook 03.
#  Convenção idêntica à do pipeline: FREQUENCIA_COMPRAS > min_compras
#  (EXCLUSIVO — min_compras=2 mantém quem tem 3 ou mais compras).
# ══════════════════════════════════════════════════════════════════════════════
LIMITE_ALVO = 0.20   # mesmo limite usado para rotular risco no notebook 01

# O universo é o do modelo: clientes de core business.
base = df[df["CORE_BUSINESS"] == 1].copy()
base["_ALTO_RISCO"] = (
    (base["TAXA_ATRASO_PAGAMENTO"] > LIMITE_ALVO)
    | (base["TAXA_ATRASO_COMODATO"] > LIMITE_ALVO)
).astype(int)

GRADE_MIN_COMPRAS = [0, 1, 2, 3, 5, 8, 12, 20]

linhas = []
for m in GRADE_MIN_COMPRAS:
    sub = base[base["FREQUENCIA_COMPRAS"] > m]
    if len(sub) == 0:
        continue
    n = len(sub)
    n_pos = int(sub["_ALTO_RISCO"].sum())
    n_neg = n - n_pos

    # Clientes cuja taxa vem de um denominador minúsculo: é o ruído que o filtro
    # pretende remover. Taxa sobre 1 ou 2 parcelas só assume valores extremos.
    frac_denom_fragil = float(
        ((sub["TOTAL_PARCELAS"] <= 2) & (sub["TOTAL_PARCELAS"] > 0)).mean()
    )

    # n da classe minoritária no teste, sob o split 70/30 do notebook 03. É este
    # número que governa a largura de TODOS os intervalos de confiança adiante:
    # abaixo de ~20, a métrica de holdout vira quase anedota.
    n_min_teste = int(min(n_pos, n_neg) * 0.30)

    linhas.append({
        "min_compras": m,
        "clientes": n,
        "% da base": n / len(base) * 100,
        "positivos": n_pos,
        "negativos": n_neg,
        "prevalência %": n_pos / n * 100,
        "denom. frágil %": frac_denom_fragil * 100,
        "n minoria no teste": n_min_teste,
        "mediana compras": float(sub["FREQUENCIA_COMPRAS"].median()),
    })

df_eleg = pd.DataFrame(linhas)

display(
    df_eleg.style
    .format({"% da base": "{:.1f}%", "prevalência %": "{:.1f}%",
             "denom. frágil %": "{:.1f}%", "mediana compras": "{:.0f}"})
    .background_gradient(subset=["clientes"], cmap="Blues")
    .background_gradient(subset=["denom. frágil %"], cmap="Reds")
    .hide(axis="index")
    .set_caption("3.7 · Efeito de min_compras sobre a amostra de modelagem")
)

In [0]:
# ─── 3.7b Visualização do trade-off ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("3.7 · O custo de cada corte de elegibilidade",
             fontsize=15, fontweight="bold")

x = df_eleg["min_compras"]

# (a) Quantos clientes sobram — o custo direto do filtro
ax = axes[0]
ax.plot(x, df_eleg["clientes"], marker="o", lw=2.4, color=COR_PRIMARIA)
ax.fill_between(x, 0, df_eleg["clientes"], alpha=0.15, color=COR_PRIMARIA)
ax.set_xlabel("min_compras (exclusivo)")
ax.set_ylabel("clientes elegíveis")
ax.set_title("Tamanho da amostra", fontsize=12, fontweight="bold")
ax.grid(alpha=0.3)
for xi, yi in zip(x, df_eleg["clientes"]):
    ax.annotate(f"{int(yi)}", (xi, yi), fontsize=7.5, ha="center", va="bottom")

# (b) Prevalência e ruído de denominador — o benefício do filtro
ax = axes[1]
ax.plot(x, df_eleg["prevalência %"], marker="o", lw=2.4,
        color=COR_ALERTA, label="prevalência do alvo")
ax.plot(x, df_eleg["denom. frágil %"], marker="s", lw=2.4, ls="--",
        color="#8E44AD", label="clientes com ≤2 parcelas")
ax.set_xlabel("min_compras (exclusivo)")
ax.set_ylabel("% dos elegíveis")
ax.set_title("Prevalência e ruído de denominador", fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.legend(fontsize=8.5)
ax.grid(alpha=0.3)

# (c) n da classe minoritária no teste — o que governa a largura dos ICs
ax = axes[2]
cores_barra = ["#E74C3C" if v < 20 else "#F39C12" if v < 40 else "#2ECC71"
               for v in df_eleg["n minoria no teste"]]
ax.bar(x, df_eleg["n minoria no teste"], color=cores_barra, alpha=0.9, width=0.7)
ax.axhline(20, ls="--", color="#E74C3C", lw=1.3,
           label="piso prático (n=20)")
ax.set_xlabel("min_compras (exclusivo)")
ax.set_ylabel("n da classe minoritária no teste")
ax.set_title("Sustentação estatística do holdout", fontsize=12, fontweight="bold")
ax.legend(fontsize=8.5)
ax.grid(axis="y", alpha=0.3)
for xi, yi in zip(x, df_eleg["n minoria no teste"]):
    ax.annotate(f"{int(yi)}", (xi, yi), fontsize=8, ha="center", va="bottom",
                fontweight="bold")

plt.tight_layout()
plt.show()

# ── Recomendação: o maior corte que ainda sustenta o holdout ─────────────────
# Regra deliberadamente conservadora: preferimos o filtro mais forte que ainda
# deixa ~20 casos da classe rara no teste. Abaixo disso, um único cliente
# reclassificado move a specificity vários pontos e a métrica vira ruído.
_viaveis = df_eleg[df_eleg["n minoria no teste"] >= 20]
print("=" * 78)
print(f"{'LEITURA PARA O NOTEBOOK 03':^78}")
print("=" * 78)
if len(_viaveis):
    _rec = _viaveis.iloc[-1]
    print(f"\n  Sugestão: min_compras = {int(_rec['min_compras'])}")
    print(f"     ├─ {int(_rec['clientes']):,} clientes elegíveis "
          f"({_rec['% da base']:.0f}% do core business)")
    print(f"     ├─ prevalência do alvo : {_rec['prevalência %']:.1f}%")
    print(f"     ├─ minoria no teste    : {int(_rec['n minoria no teste'])} casos")
    print(f"     └─ ruído de denominador: {_rec['denom. frágil %']:.1f}% com ≤2 parcelas")
else:
    _rec = df_eleg.iloc[0]
    print(f"\n  ⚠️  Nenhum corte deixa 20+ casos da classe rara no teste.")
    print(f"      A amostra é pequena demais para holdout confiável — as métricas")
    print(f"      de teste devem ser lidas com os intervalos de confiança sempre")
    print(f"      à vista, e a seleção de campeão deve usar 'cv', não 'teste'.")
    print(f"      Menor corte disponível: min_compras = {int(_rec['min_compras'])}")

print(f"\n{'─'*78}")
print("  ➜ Esta é uma leitura ESTRUTURAL dos dados, não do modelo.")
print("     O notebook 04 varre esta mesma grade como experimento do MLflow")
print("     (ESTUDO='elegibilidade') e mede o efeito sobre a métrica de validação.")
print("     Use os dois: aqui você vê o custo em amostra; lá, o ganho em performance.")
print("=" * 78)

## 📋 4. Síntese: o que a EDA entrega ao notebook 03

Cada conclusão abaixo vira uma decisão concreta no painel de hiperparâmetros.

In [0]:
# ─── Síntese das decisões que a EDA informa ───────────────────────────────────
print("=" * 84)
print(f"{'📋 SÍNTESE DA EDA · DECISÕES PARA O NOTEBOOK 03':^84}")
print("=" * 84)
print(f"  Versão dos dados analisada: {DATA_VERSION_LIDA}   (hash {INGESTAO_HASH_LIDO})")
print("=" * 84)

_universo = int(df["CORE_BUSINESS"].sum())
_prev_ou = float(((df["RISCO_FINANCEIRO"] == 1) | (df["RISCO_COMODATO"] == 1)).mean())

decisoes = [
    ("Unidade de análise",
     "ID_PESSOA",
     "vendas e comodato têm ID_PEDIDO independentes no ERP (razão fiscal)"),

    ("Universo de modelagem",
     f"CORE_BUSINESS == 1  ({_universo:,} clientes)",
     "o escopo do projeto é chopp e chopeira, não a operação inteira"),

    ("SEGMENTO como feature",
     "NÃO usar",
     "cadastro comprovadamente ruidoso (seção 3.1)"),

    ("PERFIL · CIDADE · PAGAMENTO",
     "usar como categóricas",
     "a taxa de risco varia entre categorias (seção 3.3)"),

    ("Recência como feature",
     f"usar  (Q4−Q1 = {_delta_fin:+.0f}p no risco financeiro)",
     "sinal próprio, sem parentesco aritmético com o alvo (seção 3.5)"),

    ("Combinador do alvo",
     f"OU  →  prevalência {_prev_ou:.1%}",
     f"lift de co-ocorrência {lift:.2f}× — trilhas quase independentes (3.6)"),

    ("min_compras",
     f"{int(_rec['min_compras'])} como ponto de partida",
     "maior corte que ainda sustenta o holdout (seção 3.7)"),

    ("Leitura das métricas",
     "sempre contra baseline + IC",
     f"classe positiva é maioria ({_prev_ou:.0%}) — accuracy e AUC enganam"),
]

print(f"\n  {'Decisão':<28} {'Escolha':<38} Fundamento")
print(f"  {'─'*28} {'─'*38} {'─'*30}")
for dec, esc, fund in decisoes:
    print(f"  {dec:<28} {esc:<38} {fund}")

print("\n" + "=" * 84)
print("  ⚠️  LIMITAÇÕES QUE ACOMPANHAM ESTES DADOS")
print("=" * 84)
print("  1. Sem corte temporal: features e alvo são calculados sobre a MESMA janela.")
print("     O desenho correto exigiria features até uma data de corte e alvo observado")
print("     depois dela. Não aplicado por tamanho de amostra — mas isso significa que")
print("     as métricas medem a capacidade de REPRODUZIR a regra de negócio, não de")
print("     prever o futuro.")
print()
print("  2. Features derivadas do alvo: MEDIA_DIAS_ATRASO_* compartilham origem")
print("     aritmética com TAXA_ATRASO_*, que constrói ALTO_RISCO. O notebook 03")
print("     oferece o experimento de ablação que remove essas features.")
print()
print("  3. Classe positiva majoritária: prever 'todos são risco' já acerta a maior")
print("     parte. Nenhuma métrica significa nada sem o baseline ao lado.")
print("=" * 84)
print("\n  ➜ PRÓXIMO PASSO: notebook 03, com TABELA_ENTRADA =")
print(f"       \"{TABELA_ENTRADA}\"")
print("=" * 84)